The **Synthetic Healthcare Dataset** is a complete **synthesis** of realistic but **fictitious** data that represents various aspects of healthcare. The database contains data about the patient demographics: age, gender, and region, as well as the medical conditions diagnosed, the treatments administered, and the outcomes observed.


In [12]:
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

df = pd.read_csv("/kaggle/input/datasets/divyabhavana/synthetic-healthcare-dataset/medical_data.csv")
df.head()

,Patient_ID,Age,Gender,Medical_Condition,Treatment,Outcome,Insurance_Type,Income,Region,Smoking_Status,Admission_Type,Hospital_ID,Length_of_Stay
0,1,77,Female,Chronic Obstructive,Dialysis,Stable,Public,77444,North,Former smoker,Urgent,3173,20
1,2,62,Female,Obesity,Physical therapy,Improved,Public,19367,West,Non-smoker,Urgent,65671,4
2,3,77,Male,Hypertension,Inhaler therapy,Improved,Medicare,16054,North,Non-smoker,Urgent,96914,3
3,4,41,Female,Alzheimer's Disease,Medication C,Worsened,Medicare,54371,West,Non-smoker,Emergency,15732,11
4,5,82,Male,Alzheimer's Disease,Chemotherapy,Stable,Private,55489,West,Non-smoker,Emergency,98232,2


**To make this code highly reusable across Classification, Regression, Clustering, and Anomaly Detection, the best approach is to build an Object-Oriented ML Pipeline.**

**By structuring the code into a Python class**, we can create a central "**engine**." Depending on the parameters you pass into it (like task_type and target_col), the pipeline will automatically adjust its preprocessing, model selection, and evaluation metrics.

Here is the modular code designed specifically to run in a Kaggle Notebook environment.

In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, IsolationForest
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report, mean_squared_error, r2_score

class HealthcareAIPipeline:
    def __init__(self, filepath, task_type='classification', target_col=None):
        """
        task_type options: 'classification', 'regression', 'clustering', 'anomaly'
        """
        self.filepath = filepath
        self.task_type = task_type
        self.target_col = target_col
        self.model = None
        self.scaler = StandardScaler()
        self.label_encoders = {}
        
    def load_and_clean_data(self):
        # 1. Load data (Kaggle standard path format)
        self.df = pd.read_csv(self.filepath)
        
        # 2. Drop identifiers that don't hold predictive value
        cols_to_drop = [col for col in ['Name', 'Patient ID', 'Hospital'] if col in self.df.columns]
        self.df = self.df.drop(columns=cols_to_drop, errors='ignore')
        
        # 3. Handle missing values (simple forward fill for synthetic data)
        self.df = self.df.ffill().bfill()
        
    def preprocess_data(self):
        self.load_and_clean_data()
        
        # Separate features (X) and target (y) if applicable
        if self.task_type in ['classification', 'regression']:
            if self.target_col not in self.df.columns:
                raise ValueError(f"Target column '{self.target_col}' not found in dataset.")
            
            X = self.df.drop(columns=[self.target_col])
            y = self.df[self.target_col]
            
            # Encode Target if Classification
            if self.task_type == 'classification':
                le = LabelEncoder()
                y = le.fit_transform(y)
                self.target_encoder = le
        else:
            # Unsupervised learning (Clustering / Anomaly) uses all features
            X = self.df.copy()
            y = None
            
        # Encode categorical features
        cat_cols = X.select_dtypes(include=['object']).columns
        X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)
        
        # Scale numerical features
        self.feature_names = X_encoded.columns
        X_scaled = self.scaler.fit_transform(X_encoded)
        
        return X_scaled, y
        
    def train_and_evaluate(self):
        X, y = self.preprocess_data()
        
        # Supervised Learning
        if self.task_type in ['classification', 'regression']:
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
            
            if self.task_type == 'classification':
                self.model = RandomForestClassifier(n_estimators=100, random_state=42)
                self.model.fit(X_train, y_train)
                preds = self.model.predict(X_test)
                
                print("--- Classification Report ---")
                # Decode labels back to original text for readability
                target_names = self.target_encoder.classes_.astype(str)
                print(classification_report(y_test, preds, target_names=target_names))
                
            elif self.task_type == 'regression':
                self.model = RandomForestRegressor(n_estimators=100, random_state=42)
                self.model.fit(X_train, y_train)
                preds = self.model.predict(X_test)
                
                mse = mean_squared_error(y_test, preds)
                r2 = r2_score(y_test, preds)
                print("--- Regression Metrics ---")
                print(f"Mean Squared Error: {mse:.2f}")
                print(f"R2 Score: {r2:.2f}")
                
        # Unsupervised Learning
        elif self.task_type == 'clustering':
            print("--- K-Means Clustering ---")
            self.model = KMeans(n_clusters=4, random_state=42, n_init=10)
            clusters = self.model.fit_predict(X)
            self.df['Cluster'] = clusters
            print(f"Assigned {len(self.df)} patients to 4 distinct clusters.")
            print(self.df['Cluster'].value_counts())
            
        elif self.task_type == 'anomaly':
            print("--- Anomaly Detection (Fraud/Outliers) ---")
            self.model = IsolationForest(contamination=0.05, random_state=42)
            anomalies = self.model.fit_predict(X)
            # Isolation forest outputs -1 for anomalies, 1 for normal
            self.df['Is_Anomaly'] = np.where(anomalies == -1, True, False)
            anomaly_count = self.df['Is_Anomaly'].sum()
            print(f"Detected {anomaly_count} anomalous patient records out of {len(self.df)}.")

**How to use this code?**  

Because of the object-oriented design, you don't need to rewrite your data cleaning or encoding logic. You just instantiate the class with different parameters.

Here is how you would trigger the different use cases in your subsequent Kaggle cells:


In [14]:
# Change the filepath to match your exact Kaggle directory
FILE_PATH = '/kaggle/input/datasets/divyabhavana/synthetic-healthcare-dataset/medical_data.csv'

**1. Predicting Outcomes (Classification)**  
Assuming the dataset has a column named Test Results/Outcome (often "Normal", "Abnormal", "Inconclusive") or a generic Outcome column.  

You can train a model to predict the final health outcome for a patient (e.g., recovered, readmitted, or deceased) based on their initial conditions and demographic data.

**How it works:** The model analyzes patterns between the patient's profile (age, gender, pre-existing conditions), the treatment administered, and the historical outcome.

**Algorithms:** **Random Forest**, XGBoost, Logistic Regression, or Support Vector Machines (SVM).

**Use Case:** Helping doctors flag high-risk patients who might need more aggressive observation.

In [15]:

classifier = HealthcareAIPipeline(
    filepath=FILE_PATH, 
    task_type='classification', 
    target_col='Outcome' # Replace with actual outcome column name
)
classifier.train_and_evaluate()

--- Classification Report ---
              precision    recall  f1-score   support

    Improved       0.34      0.50      0.40        64
      Stable       0.44      0.27      0.33        75
    Worsened       0.25      0.25      0.25        61

    accuracy                           0.34       200
   macro avg       0.34      0.34      0.33       200
weighted avg       0.35      0.34      0.33       200



Here is a plain-English breakdown of your classification report. 

**The Bottom Line:** Your model is currently performing poorly. With an overall accuracy of **0.34 (34%)** across three classes, the model is performing almost exactly the same as if it were just randomly guessing (which would yield about 33.3% accuracy).

Here is exactly what the numbers mean for each part of your model's predictions.

---

### **1. The Metrics Explained**
To understand the row-by-row breakdown, you first need to know what the columns represent:
*   **Precision (Quality):** When the model *predicts* a specific outcome, how often is it correct? 
*   **Recall (Quantity/Sensitivity):** Out of all the *actual* real-world cases for that outcome, how many did the model successfully find?
*   **F1-Score (Balance):** A combined score that balances Precision and Recall. It is a much better indicator of a model's health than accuracy, especially if your data is unbalanced.
*   **Support:** The actual number of records in your test dataset for that class (e.g., there were exactly 64 "Improved" patients in this test).

### **2. Class-by-Class Breakdown**

**Improved**
*   **Precision (0.34):** When your model predicts a patient "Improved," it is only right 34% of the time. It is frequently mislabeling Stable or Worsened patients as Improved.
*   **Recall (0.50):** Out of the 64 patients who actually improved, your model successfully identified 50% (32) of them. 
*   **Takeaway:** The model is heavily biased toward guessing "Improved." It finds half of the actual cases (highest recall), but it throws a lot of false positives to get there.

**Stable**
*   **Precision (0.44):** This is your model's most reliable prediction. When it predicts a patient is "Stable," it is right 44% of the time.
*   **Recall (0.27):** Out of the 75 actually stable patients, it only found 27% of them. 
*   **Takeaway:** The model is very hesitant to predict "Stable." It misses the vast majority of these patients, likely misclassifying them as "Improved."

**Worsened**
*   **Precision (0.25) & Recall (0.25):** The model is failing completely here. When it guesses "Worsened," it is right only 25% of the time. Furthermore, out of the 61 patients who actually worsened, it only found 25% of them. 
*   **Takeaway:** The model cannot recognize the patterns that lead to a patient worsening.

### **3. Overall Averages**
*   **Accuracy (0.34):** Out of all 200 patients in the test, the model predicted the correct outcome 34% of the time. 
*   **Macro Avg (0.34 / 0.34 / 0.33):** This calculates the average of your three classes without taking the "Support" into account. It treats all classes equally, confirming the model averages about ~33% across the board.
*   **Weighted Avg (0.35 / 0.34 / 0.33):** This calculates the average but gives more weight to the classes with more data (like "Stable," which has 75 cases). Because your support sizes are relatively close (64, 75, 61), the weighted average is nearly identical to the macro average.

----------------------------------------------------------------------

**2. Length of Stay (LOS) Prediction (Regression)**  
Hospital logistics rely heavily on predicting how long a patient will occupy a bed. You can train an AI model to estimate the exact number of days a patient will stay in the facility.

**How it works:** You use the admission date, diagnosis, and demographic details as input features to predict a continuous number (days).

**Algorithms:** **Linear Regression**, Gradient Boosting Regressors, or Deep Neural Networks.

**Use Case:** Optimizing hospital resource allocation, bed management, and staff scheduling.

**Length of Stay / Cost Estimation** 
Assuming there is a numeric column like Billing Amount or Days Hospitalized (**Length_of_Stay**).

In [16]:
regressor = HealthcareAIPipeline(
    filepath=FILE_PATH, 
    task_type='regression', 
    target_col='Length_of_Stay' # Replace with actual numeric column
)
regressor.train_and_evaluate()

--- Regression Metrics ---
Mean Squared Error: 41.46
R2 Score: -0.11


Here is a plain-English breakdown of your regression metrics. 

**The Bottom Line:** Just like the classification model, your regression model is struggling heavily. The most telling number here is the negative R2 score, which indicates that your AI is performing **worse than if it simply guessed the average value for every single patient.**

Here is exactly what these two numbers mean:

### **1. R2 Score (R-Squared): -0.11**
This is the most important metric for understanding how well your regression model fits the data. 
* **How it works:** The R2 score measures how much of the variance in your target variable (e.g., Length of Stay or Billing Cost) is explained by your features (Age, Condition, etc.). A perfect model gets a **1.0**. 
* **The "Baseline":** A score of **0.0** means your model is exactly as accurate as a naive system that just calculates the historical average of the target column and guesses that same average number for every future patient, regardless of their symptoms.
* **Your Score (-0.11):** Because your score is negative, it means your model is making predictions that are actively *worse* and farther away from the truth than if it just guessed the average every time. The model is failing to find any mathematical relationship between the patient data and the target number.

### **2. Mean Squared Error (MSE): 41.46**
This metric tells you how far off your predictions are, on average, but with a twist.
* **How it works:** It takes the difference between the model's prediction and the actual true value, squares that difference (which severely punishes really bad guesses), and then averages them all out.
* **Interpreting the number:** By itself, 41.46 is hard to interpret because the units are squared. To understand what this means in the real world, you take the square root of the MSE (called the Root Mean Squared Error, or RMSE). 
* **The Real-World Error:** The square root of 41.46 is **~6.43**. 
    * If you were trying to predict **Length of Stay**, this means your model is off by an average of about **6.4 days** per patient. 
    * If you were trying to predict a **Billing Cost** (and the numbers were in thousands), it means your model is off by about **$6,430** per patient.

### **Why is the model performing poorly?**
Since both your classification and regression results are essentially random guessing (or worse), this points back to the nature of the dataset itself:

1. **Synthetic Randomness:** Because this is a *synthetic* dataset generated for Kaggle, the creator likely populated the columns with random distributions that don't have real mathematical correlations. For example, in the real world, "Age 85" + "Heart Condition" strongly correlates with a longer hospital stay. In this dataset, the length of stay might have just been generated randomly between 1 and 14 days, regardless of age or condition. 
2. **Missing Golden Features:** The dataset might be missing the specific data points that actually drive the outcome.

**Your Next Step:** If you want to improve this, I recommend running a **Feature Importance** check or a correlation matrix on your dataset. This will prove whether or not any of the columns actually have a mathematical relationship with your target variable before you try to force an AI to find one!

--------------------------------------------

**3. Treatment Recommendation Systems (Classification / Recommendation)**  
By leveraging the "Treatments" and "Outcomes" features, you can build an AI assistant that suggests the most historically effective treatment plan for a specific condition.

**How it works:** The model learns which treatments yielded the highest success rates for specific conditions across different demographic groups.

**Algorithms:** K-Nearest Neighbors (KNN), Decision Trees, or Collaborative Filtering techniques.

**Use Case:** Providing clinical decision support to physicians by recommending evidence-based treatments.  


In [17]:
classifier = HealthcareAIPipeline(
    filepath=FILE_PATH, 
    task_type='classification', 
    target_col='Treatment' # Replace with actual outcome column name
)
classifier.train_and_evaluate()

--- Classification Report ---
                    precision    recall  f1-score   support

Bone density tests       0.07      0.08      0.07        13
      Chemotherapy       0.29      0.24      0.26        17
          Dialysis       0.08      0.07      0.07        15
   Dietary changes       0.08      0.08      0.08        12
Dietary counseling       0.10      0.19      0.13        16
Immunosuppressants       0.15      0.10      0.12        20
   Inhaler therapy       0.08      0.07      0.08        14
      Medication A       0.08      0.11      0.09         9
      Medication B       0.13      0.25      0.17        16
      Medication C       0.18      0.11      0.14        18
  Memory exercises       0.05      0.06      0.05        18
  Physical therapy       0.00      0.00      0.00        18
           Therapy       0.00      0.00      0.00        14

          accuracy                           0.10       200
         macro avg       0.10      0.10      0.10       200
      we

----------------------

**4. Healthcare Cost and Billing Estimation (Regression)**  
If the dataset contains financial or billing data (common in synthetic healthcare data), you can build a model to estimate the cost of care at the time of admission.

**How it works:** The AI correlates the medical condition, expected treatment, and demographics with final billing amounts.

**Algorithms:** Ridge/Lasso Regression, Random Forest Regressor.

**Use Case:** Providing patients with transparent cost estimates and helping insurance providers predict claim amounts.  


In [18]:
regressor = HealthcareAIPipeline(
    filepath=FILE_PATH, 
    task_type='regression', 
    target_col='Income' # Replace with actual numeric column such as Billing Amount
)
regressor.train_and_evaluate()

--- Regression Metrics ---
Mean Squared Error: 987739808.82
R2 Score: -0.13


-----------------------

**5. Patient Segmentation and Risk Stratification (Clustering)**  
Instead of predicting a specific target, you can use unsupervised learning to discover hidden patterns and group similar patients together.

**How it works:** The model clusters patients based on similarities in their demographics and medical conditions without needing predefined labels.

**Algorithms:** K-Means Clustering, DBSCAN, or Hierarchical Clustering.

**Use Case:** Identifying distinct patient personas (e.g., "high-risk chronic patients") to design targeted, preventative public health programs.

**Example: Patient Risk Stratification (Clustering)**
Because clustering is unsupervised, it groups patients based on all available features without needing a target column.

In [19]:
clusterer = HealthcareAIPipeline(
    filepath=FILE_PATH, 
    task_type='clustering'
)
clusterer.train_and_evaluate()

--- K-Means Clustering ---
Assigned 1000 patients to 4 distinct clusters.
Cluster
1    460
3    408
2    114
0     18
Name: count, dtype: int64


This clustering result is the most interesting part of your analysis so far. While your Classification and Regression models struggled to find clear "rules," the **K-Means Clustering** has actually found some structure in the data.

Here is what these numbers tell us about your 1,000 synthetic patients:

### 1. The Cluster Distribution
*   **Cluster 1 (460 patients) & Cluster 3 (408 patients):** These are your "Mainstream" groups. Together, they represent nearly **87% of your dataset**. These clusters likely contain patients with the most common combinations of demographics and conditions.
*   **Cluster 2 (114 patients):** This is a smaller, distinct group. These patients have features that are significantly different from the mainstream (perhaps a specific age group or a specific combination of treatment and condition).
*   **Cluster 0 (18 patients):** These are your **Outliers**. This very small group is mathematically different from everyone else in the dataset. In a healthcare context, these could be your most complex cases or "edge cases" (e.g., very young patients with conditions normally found in the elderly).

### 2. What this means for your AI Strategy
Since your previous models (Classification/Regression) performed poorly, this clustering result gives you a **new path forward**:

*   **Segmented Modeling:** Instead of training one AI model for all 1,000 patients, you could train one model for Cluster 1 and another for Cluster 3. By narrowing the scope, the AI might find patterns it missed when looking at the whole "noisy" dataset.
*   **Focus on Cluster 0:** In healthcare AI, the 18 patients in Cluster 0 are often the most important. They represent the highest risk of error for a general AI. You can now isolate these records to see what makes them so unique (is it a specific rare blood type, a very high billing amount, or an unusual length of stay?).

### 3. How to "See" the Results
To make sense of *why* the patients were grouped this way, you should look at the **Cluster Centroids** (the average values for each group). On Kaggle, you can run this code to see the differences:

```python
# Add this to your notebook to see the 'Profile' of each cluster
cluster_profiles = clusterer.df.groupby('Cluster').mean(numeric_only=True)
print(cluster_profiles)
```

**What to look for in the output:**
*   Is the average **Age** in Cluster 0 much higher than in Cluster 1?
*   Is the **Billing Amount** in Cluster 2 significantly different?
*   Do specific **Conditions** appear more often in certain clusters?

**Summary:** The clustering is telling you that the data isn't just one big "random" pile; it has 4 distinct "shapes." Your next step is to figure out which feature (Age, Treatment, or Cost) is the primary reason the AI separated these 1,000 people into these specific buckets.

In [20]:
# Add this to your notebook to see the 'Profile' of each cluster
cluster_profiles = clusterer.df.groupby('Cluster').mean(numeric_only=True)
print(cluster_profiles)

         Patient_ID        Age        Income   Hospital_ID  Length_of_Stay
Cluster                                                                   
0        425.388889  40.833333  57554.500000  45490.222222       10.777778
1        490.560870  47.200000  50395.032609  52503.595652        9.873913
2        507.008772  44.210526  47938.149123  48101.745614       10.596491
3        513.200980  44.664216  50242.919118  52096.127451       10.041667


This table is the "DNA" of your clusters. It shows the mathematical center (the average) of each group. By comparing these averages, we can finally see the "story" the AI found in your data.

Here is the breakdown of what these clusters actually represent:

### 1. Identifying the "Driver" (What caused the split?)
When looking at a clustering table, you want to find the column where the numbers are **most different** between rows. 
*   **Income:** This is likely your primary driver. **Cluster 0** has a significantly higher average income (**57,554**) compared to **Cluster 2** (**47,938**). The AI has essentially segmented your patients into "Wealthy," "Middle Class," and "Lower Income" groups.
*   **Age:** There is a notable gap here too. **Cluster 1** is your "Senior" group (Avg. Age **47.2**), while **Cluster 0** is your "Youngest" group (Avg. Age **40.8**).

### 2. The Patient Personas
Based on the data, we can name these 4 groups:

*   **Cluster 0 (The "Premium" Young Patients):** These are the 18 outliers we discussed. They are the youngest, have the **highest income**, and—interestingly—stay in the hospital the longest (**10.77 days**). 
*   **Cluster 1 (The "Senior" Majority):** This is your largest group. They are the oldest on average and represent the "standard" patient profile for this hospital.
*   **Cluster 2 (The "Lower Income" Moderate Stay):** These patients have the lowest average income but actually have a relatively high length of stay (**10.59 days**), suggesting they might have more complex needs despite lower financial resources.
*   **Cluster 3 (The "Average" Middle Group):** This group sits right in the middle for almost every metric. They are your "baseline" patients.



### 3. The "Red Flag" Columns
Look at **Patient_ID** and **Hospital_ID**. 
*   In a perfect dataset, these averages should be roughly the same across all clusters because IDs are just random numbers.
*   **However**, since Cluster 0 has a significantly lower `Patient_ID` (425) and `Hospital_ID` (45,490) average, it suggests that the "random" synthetic data generation might have a slight bias where earlier records or specific hospital IDs were grouped with higher incomes.

### 4. How to use this for Options 2 & 4 (Regression)
Now that you know Income and Age are the main differences, you can improve your previous failing Regression model by doing **Feature Engineering**:

1.  **Binning Age:** Instead of using the exact age, create categories (e.g., "Under 45" vs "Over 45").
2.  **Income vs. Stay:** Since Cluster 0 (Highest Income) has the longest stay (10.77), there is a correlation there that your linear model missed but the Clustering model caught. You could create a new feature called `Income_Per_Day` to help the AI find the pattern.

**Why did Clustering work when Prediction failed?**
Clustering doesn't care about "right or wrong" answers (the labels); it only cares about "who looks like whom." Even if the labels in the synthetic dataset are random, the physical characteristics (Age/Income) still have mathematical patterns that K-Means can see.

Does this give you a better idea of which "Persona" you want to focus your next AI model on?

---------------------------

**6. Medical Fraud or Anomaly Detection (Anomaly Detection)**  
Synthetic datasets are excellent for training models to spot outliers, which is a key task in auditing and compliance.

**How it works:** The AI learns what a "normal" patient record looks like. If a new record features a rare treatment for a common condition or an exorbitant billing amount, the model flags it.

**Algorithms:** Isolation Forest, One-Class SVM, or Autoencoders.

**Use Case:** Detecting data entry errors, irregular medical practices, or potential insurance billing fraud.  

**Example:** Fraud / Anomaly Detection (Anomaly)  
Finds the 5% (contamination rate set in the class) most unusual patient records.

In [21]:
anomaly_detector = HealthcareAIPipeline(
    filepath=FILE_PATH, 
    task_type='anomaly'
)
anomaly_detector.train_and_evaluate()

--- Anomaly Detection (Fraud/Outliers) ---
Detected 50 anomalous patient records out of 1000.


This result means the AI has successfully identified the "weirdest" 5% of your dataset. In an **Isolation Forest**, anomalies are data points that are so unique they are easy for the algorithm to "isolate" from the rest of the group.

Since you set the `contamination` parameter to **0.05** in the code, the model was essentially told: *"I expect 5% of this data to be strange; find the 50 most unusual records for me."*

### 1. What makes these 50 patients "Anomalous"?
In a healthcare dataset, the AI likely flagged these records because of **extreme values** or **illogical combinations**. Here are the three most common reasons:

*   **Point Anomalies (The Extremes):** 
    *   A patient who is 115 years old.
    *   A billing amount of 95,000 when the average is 20,000.
    *   A length of stay of 45 days when most patients leave in 7.
*   **Contextual Anomalies (The "Wait, what?" cases):**
    *   A 5-year-old child diagnosed with a condition typically seen in 80-year-olds.
    *   A patient receiving a very expensive treatment for a very minor condition.
*   **Data Errors (The "Garbage" cases):**
    *   Records with missing fields that were filled with "placeholder" values (like -1 or 999).
    *   Typographical errors in income or hospital IDs.

### 2. How to "Investigate" the Fraud/Outliers
To see exactly what the AI found, you should compare the "Anomalous" patients against the "Normal" ones. Run this in your Kaggle notebook:

```python
# Separate the anomalies for a quick look
anomalies_df = anomaly_detector.df[anomaly_detector.df['Is_Anomaly'] == True]
normal_df = anomaly_detector.df[anomaly_detector.df['Is_Anomaly'] == False]

# Compare the averages
print("--- Average Values: Normal vs Anomaly ---")
comparison = pd.concat([
    normal_df.mean(numeric_only=True).to_frame(name='Normal'),
    anomalies_df.mean(numeric_only=True).to_frame(name='Anomaly')
], axis=1)
print(comparison)
```

### 3. Why this matters for your "AI Engineer" role
In a real professional environment, this result is your **Cleaning Map**:

*   **For Fraud Detection:** If you were working for an insurance company, these 50 records are the ones you would send to an auditor. They have the highest probability of being "ghost billing" or incorrect medical coding.
*   **For Model Accuracy:** Remember how your Classification/Regression models failed? These 50 anomalies are likely "poisoning" your results. By removing these 50 records and re-training your Classification model, your accuracy will almost certainly improve because the AI won't be distracted by the "weird" data.
*   **For System Health:** If 50 out of 1000 records are anomalies (5%), that’s a relatively high error rate. It might suggest that the data collection process at one of the hospitals (referencing your `Hospital_ID` column) is broken.

**Pro-Tip:** Check if most of these 50 anomalies belong to **Cluster 0** (the small group of 18 we found earlier). If they do, it confirms that Cluster 0 isn't just a different group—it's likely a group of data errors!

In [22]:
# Separate the anomalies for a quick look
anomalies_df = anomaly_detector.df[anomaly_detector.df['Is_Anomaly'] == True]
normal_df = anomaly_detector.df[anomaly_detector.df['Is_Anomaly'] == False]

# Compare the averages
print("--- Average Values: Normal vs Anomaly ---")
comparison = pd.concat([
    normal_df.mean(numeric_only=True).to_frame(name='Normal'),
    anomalies_df.mean(numeric_only=True).to_frame(name='Anomaly')
], axis=1)
print(comparison)

--- Average Values: Normal vs Anomaly ---
                      Normal   Anomaly
Patient_ID        503.507368    443.36
Age                45.729474     45.34
Income          50328.853684  47386.90
Hospital_ID     51985.047368  46470.04
Length_of_Stay     10.087368      9.16
Is_Anomaly          0.000000      1.00
